In [12]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('MEPS uncleaned.csv')
df

,fullName,country,politicalGroup,id,nationalPoliticalGroup
0,Mika AALTOLA,Finland,Group of the European People's Party (Christia...,256810,Kansallinen Kokoomus
1,Maravillas ABADÍA JOVER,Spain,Group of the European People's Party (Christia...,257043,Partido Popular
2,Magdalena ADAMOWICZ,Poland,Group of the European People's Party (Christia...,197490,Independent
3,Georgios AFTIAS,Greece,Group of the European People's Party (Christia...,256820,Nea Demokratia
4,Oihane AGIRREGOITIA MARTÍNEZ,Spain,Renew Europe Group,256987,Partido Nacionalista Vasco
...,...,...,...,...,...
714,Nicola ZINGARETTI,Italy,Group of the Progressive Alliance of Socialist...,28419,Partito Democratico
715,Kosma ZŁOTOWSKI,Poland,European Conservatives and Reformists Group,124884,Prawo i Sprawiedliwość
716,Juan Ignacio ZOIDO ÁLVAREZ,Spain,Group of the European People's Party (Christia...,197621,Partido Popular
717,Željana ZOVKO,Croatia,Group of the European People's Party (Christia...,185341,Hrvatska demokratska zajednica


In [13]:
df.rename(columns={'fullName': 'Full Name'}, inplace=True)
df.rename(columns={'country': 'Country'}, inplace=True)
df.rename(columns={'id': 'Member ID'}, inplace=True)
df.rename(columns={'politicalGroup': 'EU Political Group'}, inplace=True)
df.rename(columns={'nationalPoliticalGroup': 'National Political Group'}, inplace=True)


In [14]:
# Add first name and last name columns. Account for spaces.

In [15]:
import pandas as pd

# Matches: "everything" + " " + "ALLCAPS tail (possibly multiple tokens)"
# Uses Unicode-aware uppercase detection via [^\W\d_]=letters (no digits/underscore)
pattern = r'^(?P<given>.+?)\s+(?P<surname>(?:[^\W\d_]+(?:[-\'’][^\W\d_]+)?)(?:\s+(?:[^\W\d_]+(?:[-\'’][^\W\d_]+)?))*)$'

# Helper: True if token is "all caps" in a Unicode-safe way
def is_all_caps_token(tok: str) -> bool:
    # keep letters + internal hyphen/apostrophe; ignore punctuation around
    core = tok.strip(" .,:;()[]{}")
    # require at least one cased letter and all cased letters uppercase
    return any(ch.isalpha() and ch.isupper() for ch in core) and not any(ch.isalpha() and ch.islower() for ch in core)

def split_standard_caps(full: str):
    if pd.isna(full) or not str(full).strip():
        return pd.NA, pd.NA

    parts = str(full).strip().split()

    # Walk from end collecting ALL-CAPS tokens
    i = len(parts) - 1
    surname_parts = []
    while i >= 0 and is_all_caps_token(parts[i]):
        surname_parts.append(parts[i])
        i -= 1

    if not surname_parts:
        # fallback: no caps tail found
        return " ".join(parts), pd.NA

    surname = " ".join(reversed(surname_parts))
    given = " ".join(parts[:i+1]) if i >= 0 else pd.NA
    return given, surname

df[["Given Names", "Surnames"]] = (
    df["Full Name"]
      .apply(split_standard_caps)
      .apply(pd.Series)
)


In [16]:
df

,Full Name,Country,EU Political Group,Member ID,National Political Group,Given Names,Surnames
0,Mika AALTOLA,Finland,Group of the European People's Party (Christia...,256810,Kansallinen Kokoomus,Mika,AALTOLA
1,Maravillas ABADÍA JOVER,Spain,Group of the European People's Party (Christia...,257043,Partido Popular,Maravillas,ABADÍA JOVER
2,Magdalena ADAMOWICZ,Poland,Group of the European People's Party (Christia...,197490,Independent,Magdalena,ADAMOWICZ
3,Georgios AFTIAS,Greece,Group of the European People's Party (Christia...,256820,Nea Demokratia,Georgios,AFTIAS
4,Oihane AGIRREGOITIA MARTÍNEZ,Spain,Renew Europe Group,256987,Partido Nacionalista Vasco,Oihane,AGIRREGOITIA MARTÍNEZ
...,...,...,...,...,...,...,...
714,Nicola ZINGARETTI,Italy,Group of the Progressive Alliance of Socialist...,28419,Partito Democratico,Nicola,ZINGARETTI
715,Kosma ZŁOTOWSKI,Poland,European Conservatives and Reformists Group,124884,Prawo i Sprawiedliwość,Kosma,ZŁOTOWSKI
716,Juan Ignacio ZOIDO ÁLVAREZ,Spain,Group of the European People's Party (Christia...,197621,Partido Popular,Juan Ignacio,ZOIDO ÁLVAREZ
717,Željana ZOVKO,Croatia,Group of the European People's Party (Christia...,185341,Hrvatska demokratska zajednica,Željana,ZOVKO


In [17]:
df = df[['Member ID', 'Country', 'Full Name', 'Given Names', 'Surnames', 'EU Political Group', 'National Political Group']]

In [18]:
df

,Member ID,Country,Full Name,Given Names,Surnames,EU Political Group,National Political Group
0,256810,Finland,Mika AALTOLA,Mika,AALTOLA,Group of the European People's Party (Christia...,Kansallinen Kokoomus
1,257043,Spain,Maravillas ABADÍA JOVER,Maravillas,ABADÍA JOVER,Group of the European People's Party (Christia...,Partido Popular
2,197490,Poland,Magdalena ADAMOWICZ,Magdalena,ADAMOWICZ,Group of the European People's Party (Christia...,Independent
3,256820,Greece,Georgios AFTIAS,Georgios,AFTIAS,Group of the European People's Party (Christia...,Nea Demokratia
4,256987,Spain,Oihane AGIRREGOITIA MARTÍNEZ,Oihane,AGIRREGOITIA MARTÍNEZ,Renew Europe Group,Partido Nacionalista Vasco
...,...,...,...,...,...,...,...
714,28419,Italy,Nicola ZINGARETTI,Nicola,ZINGARETTI,Group of the Progressive Alliance of Socialist...,Partito Democratico
715,124884,Poland,Kosma ZŁOTOWSKI,Kosma,ZŁOTOWSKI,European Conservatives and Reformists Group,Prawo i Sprawiedliwość
716,197621,Spain,Juan Ignacio ZOIDO ÁLVAREZ,Juan Ignacio,ZOIDO ÁLVAREZ,Group of the European People's Party (Christia...,Partido Popular
717,185341,Croatia,Željana ZOVKO,Željana,ZOVKO,Group of the European People's Party (Christia...,Hrvatska demokratska zajednica


In [19]:
df['Multi-Part Given Name'] = df['Given Names'].str.contains(r'\s+', na=False).astype(int)
df['Multi-Part Surname'] = df['Surnames'].str.contains(r'\s+', na=False).astype(int)


/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_7818/888490398.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Multi-Part Given Name'] = df['Given Names'].str.contains(r'\s+', na=False).astype(int)
/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_7818/888490398.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Multi-Part Surname'] = df['Surnames'].str.contains(r'\s+', na=False).astype(int)


In [20]:
df

,Member ID,Country,Full Name,Given Names,Surnames,EU Political Group,National Political Group,Multi-Part Given Name,Multi-Part Surname
0,256810,Finland,Mika AALTOLA,Mika,AALTOLA,Group of the European People's Party (Christia...,Kansallinen Kokoomus,0,0
1,257043,Spain,Maravillas ABADÍA JOVER,Maravillas,ABADÍA JOVER,Group of the European People's Party (Christia...,Partido Popular,0,1
2,197490,Poland,Magdalena ADAMOWICZ,Magdalena,ADAMOWICZ,Group of the European People's Party (Christia...,Independent,0,0
3,256820,Greece,Georgios AFTIAS,Georgios,AFTIAS,Group of the European People's Party (Christia...,Nea Demokratia,0,0
4,256987,Spain,Oihane AGIRREGOITIA MARTÍNEZ,Oihane,AGIRREGOITIA MARTÍNEZ,Renew Europe Group,Partido Nacionalista Vasco,0,1
...,...,...,...,...,...,...,...,...,...
714,28419,Italy,Nicola ZINGARETTI,Nicola,ZINGARETTI,Group of the Progressive Alliance of Socialist...,Partito Democratico,0,0
715,124884,Poland,Kosma ZŁOTOWSKI,Kosma,ZŁOTOWSKI,European Conservatives and Reformists Group,Prawo i Sprawiedliwość,0,0
716,197621,Spain,Juan Ignacio ZOIDO ÁLVAREZ,Juan Ignacio,ZOIDO ÁLVAREZ,Group of the European People's Party (Christia...,Partido Popular,1,1
717,185341,Croatia,Željana ZOVKO,Željana,ZOVKO,Group of the European People's Party (Christia...,Hrvatska demokratska zajednica,0,0


In [21]:
df.to_csv('MEPS cleaned.csv', index=False)

In [22]:
cols_to_encode = ['Country', 'EU Political Group', 'National Political Group']

df_encoded = pd.get_dummies(df, columns=cols_to_encode, dtype=int)


In [23]:
df_encoded

,Member ID,Full Name,Given Names,Surnames,Multi-Part Given Name,Multi-Part Surname,Country_Austria,Country_Belgium,Country_Bulgaria,Country_Croatia,...,National Political Group_Vihreä liitto,National Political Group_Vlaams Belang,National Political Group_Volkspartij voor Vrijheid en Democratie,National Political Group_Volt,National Political Group_Vooruit,National Political Group_Vänsterpartiet,National Political Group_We continue the change – Democratic Bulgaria,National Political Group_Ökologisch-Demokratische Partei,National Political Group_Österreichische Volkspartei,National Political Group_Εθνικό Λαϊκό Μέτωπο
0,256810,Mika AALTOLA,Mika,AALTOLA,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,257043,Maravillas ABADÍA JOVER,Maravillas,ABADÍA JOVER,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,197490,Magdalena ADAMOWICZ,Magdalena,ADAMOWICZ,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,256820,Georgios AFTIAS,Georgios,AFTIAS,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,256987,Oihane AGIRREGOITIA MARTÍNEZ,Oihane,AGIRREGOITIA MARTÍNEZ,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714,28419,Nicola ZINGARETTI,Nicola,ZINGARETTI,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
715,124884,Kosma ZŁOTOWSKI,Kosma,ZŁOTOWSKI,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
716,197621,Juan Ignacio ZOIDO ÁLVAREZ,Juan Ignacio,ZOIDO ÁLVAREZ,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
717,185341,Željana ZOVKO,Željana,ZOVKO,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [24]:
df_encoded.to_csv('MEPS cleaned with all dummies.csv', index=False)

In [10]:
df['id'].nunique()

719

In [4]:
df['country'].value_counts()

country
Germany        96
France         81
Italy          76
Spain          60
Poland         53
Romania        33
Netherlands    31
Belgium        22
Greece         21
Sweden         21
Czechia        21
Hungary        21
Portugal       21
Austria        20
Bulgaria       17
Denmark        15
Slovakia       15
Finland        15
Ireland        14
Croatia        12
Lithuania      11
Slovenia        9
Latvia          9
Estonia         7
Luxembourg      6
Cyprus          6
Malta           6
Name: count, dtype: int64

In [1]:
counts_by_group = df.groupby('politicalGroup')['id'].nunique()
counts_by_group

NameError: name 'df' is not defined

In [12]:
counts_by_group = df.groupby('politicalGroup')['nationalPoliticalGroup'].nunique()
counts_by_group

politicalGroup
Europe of Sovereign Nations Group                                                            8
European Conservatives and Reformists Group                                                 23
Group of the European People's Party (Christian Democrats)                                  47
Group of the Greens/European Free Alliance                                                  22
Group of the Progressive Alliance of Socialists and Democrats in the European Parliament    29
Non-attached Members                                                                        19
Patriots for Europe Group                                                                   16
Renew Europe Group                                                                          37
The Left group in the European Parliament - GUE/NGL                                         19
Name: nationalPoliticalGroup, dtype: int64